In [ ]:
%%sql -r dataframe_1
-- Create database, schemas, and warehouse
CREATE OR REPLACE DATABASE DATASET_GENOME;

CREATE OR REPLACE SCHEMA DATASET_GENOME.BRONZE;
CREATE OR REPLACE SCHEMA DATASET_GENOME.SILVER;
CREATE OR REPLACE SCHEMA DATASET_GENOME.GOLD;

CREATE OR REPLACE WAREHOUSE DATASET_GENOME_WH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;

USE WAREHOUSE DATASET_GENOME_WH;
USE DATABASE DATASET_GENOME;
USE SCHEMA BRONZE;

In [ ]:
-- Verify setup
SHOW SCHEMAS IN DATABASE DATASET_GENOME;
SHOW WAREHOUSES LIKE 'DATASET_GENOME_WH';

In [ ]:
# Imports and configuration
import numpy as np
import pandas as pd
import random
import math
import json
from scipy.spatial.distance import cosine, euclidean
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import (
    IntegerType, LongType, DoubleType, FloatType,
    DecimalType, ShortType, StringType, TimestampType, DateType
)

session = get_active_session()

GLOBAL_SEED = 1223
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

session.sql("USE WAREHOUSE DATASET_GENOME_WH").collect()
session.sql("USE DATABASE DATASET_GENOME").collect()
session.sql("USE SCHEMA BRONZE").collect()

print("Snowflake session ready")
print("Database:", session.get_current_database())
print("Schema:", session.get_current_schema())
print("Warehouse:", session.get_current_warehouse())
print("Global seed:", GLOBAL_SEED)

In [ ]:
# Generate base dataset
N_ROWS = 20000

def generate_base_dataset(n_rows=20000, seed=1223):
    rng = np.random.RandomState(seed)

    age = rng.normal(40, 12, n_rows).clip(18, 90).round(0)
    income = rng.lognormal(mean=10.5, sigma=0.6, size=n_rows).round(2)
    account_balance = rng.normal(5000, 2000, n_rows).clip(0).round(2)
    transaction_amount = rng.lognormal(mean=4, sigma=1.2, size=n_rows).round(2)
    credit_score = rng.normal(650, 80, n_rows).clip(300, 850).round(0)
    risk_score = (0.4 * (credit_score - 300) / 550 + 0.3 * np.log1p(income) / 12 + rng.normal(0, 0.05, n_rows)).clip(0, 1).round(4)
    default_flag = (risk_score > 0.65).astype(int)
    tenure_months = rng.randint(1, 120, n_rows)
    num_products = rng.randint(1, 6, n_rows)
    num_transactions = rng.poisson(15, n_rows)
    avg_transaction_value = (transaction_amount / np.maximum(num_transactions, 1)).round(2)
    balance_change = rng.normal(0, 500, n_rows).round(2)
    interest_rate = rng.normal(5.5, 1.5, n_rows).clip(0.5, 20).round(2)
    loan_amount = rng.lognormal(mean=9, sigma=1.0, size=n_rows).round(2)
    monthly_payment = (loan_amount / 60).round(2)
    debt_to_income = (monthly_payment / np.maximum(income / 12, 1)).round(4)

    gender = rng.choice(["M", "F", "Other"], n_rows, p=[0.48, 0.48, 0.04])
    region = rng.choice(["North", "South", "East", "West", "Central"], n_rows, p=[0.25, 0.2, 0.2, 0.2, 0.15])
    customer_segment = rng.choice(["Bronze", "Silver", "Gold", "Platinum"], n_rows, p=[0.4, 0.3, 0.2, 0.1])
    employment_type = rng.choice(["Salaried", "Self-Employed", "Business", "Retired"], n_rows, p=[0.5, 0.2, 0.2, 0.1])
    education = rng.choice(["HighSchool", "Bachelor", "Master", "PhD"], n_rows, p=[0.3, 0.4, 0.2, 0.1])
    marital_status = rng.choice(["Single", "Married", "Divorced"], n_rows, p=[0.35, 0.5, 0.15])
    channel = rng.choice(["WEB", "MOBILE", "BRANCH", "CALL"], n_rows, p=[0.4, 0.3, 0.2, 0.1])

    base_date = pd.Timestamp("2023-01-01")
    dates = pd.to_datetime(base_date + pd.to_timedelta(rng.randint(0, 365, n_rows), unit="D"))
    last_activity_days = rng.randint(1, 365, n_rows)

    df = pd.DataFrame({
        "customer_id": [f"CUST{i:07d}" for i in range(n_rows)],
        "age": age.astype(int),
        "income": income,
        "account_balance": account_balance,
        "transaction_amount": transaction_amount,
        "credit_score": credit_score.astype(int),
        "risk_score": risk_score,
        "default_flag": default_flag,
        "tenure_months": tenure_months,
        "num_products": num_products,
        "num_transactions": num_transactions,
        "avg_transaction_value": avg_transaction_value,
        "balance_change": balance_change,
        "interest_rate": interest_rate,
        "loan_amount": loan_amount,
        "monthly_payment": monthly_payment,
        "debt_to_income": debt_to_income,
        "gender": gender,
        "region": region,
        "customer_segment": customer_segment,
        "employment_type": employment_type,
        "education": education,
        "marital_status": marital_status,
        "channel": channel,
        "transaction_date": dates,
        "last_activity_days": last_activity_days,
    })
    return df

base_df = generate_base_dataset(N_ROWS, GLOBAL_SEED)
print("Base dataset shape:", base_df.shape)
print("Columns:", list(base_df.columns))

In [ ]:
# Generate all 12 variants
def make_variants(base_df, seed=1223):
    rng = np.random.RandomState(seed)
    variants = {}

    variants["G01_BASE"] = base_df.copy()

    variants["G02_ROW_SHUFFLED"] = base_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    cols = list(base_df.columns)
    shuffled_cols = cols[:5] + cols[10:15] + cols[5:10] + cols[15:]
    variants["G03_COLUMN_REORDERED"] = base_df[shuffled_cols].copy()

    rename_map = {c: f"col_{i}" for i, c in enumerate(base_df.columns)}
    variants["G04_COLUMN_RENAMED"] = base_df.rename(columns=rename_map).copy()

    df5 = base_df.copy()
    for col in df5.columns:
        mask = rng.random(len(df5)) < 0.20
        df5.loc[mask, col] = np.nan
    variants["G05_MISSING_20"] = df5

    df6 = base_df.copy()
    for col in df6.select_dtypes(include=[np.number]).columns:
        df6[col] = df6[col] + rng.normal(0, 0.10 * df6[col].std(), len(df6))
    variants["G06_NOISE_10"] = df6

    df7 = base_df.copy()
    dup_idx = rng.choice(len(df7), size=int(0.20 * len(df7)), replace=False)
    df7 = pd.concat([df7, df7.iloc[dup_idx]], ignore_index=True)
    variants["G07_REDUNDANCY_20"] = df7

    df8 = base_df.copy()
    n_min = int(0.10 * len(df8))
    n_maj = len(df8) - n_min
    maj = df8[df8["default_flag"] == 0].sample(n=n_maj, replace=True, random_state=seed)
    mino = df8[df8["default_flag"] == 1].sample(n=n_min, replace=True, random_state=seed)
    df8 = pd.concat([maj, mino], ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    variants["G08_IMBALANCE_90"] = df8

    df9 = base_df.copy()
    for col in df9.select_dtypes(include=[np.number]).columns:
        df9[col] = df9[col] + 0.20 * df9[col].std()
    variants["G09_DRIFT_20"] = df9

    df10 = base_df.copy()
    df10["transaction_amount"] = rng.lognormal(mean=4, sigma=1.2, size=len(df10)).round(2)
    variants["G10_DEPENDENCY_BREAK"] = df10

    df11 = base_df.copy()
    for col in df11.columns[:5]:
        df11.loc[rng.random(len(df11)) < 0.15, col] = np.nan
    for col in df11.select_dtypes(include=[np.number]).columns[:5]:
        df11[col] = df11[col] + rng.normal(0, 0.05 * df11[col].std(), len(df11))
    dup_idx = rng.choice(len(df11), size=int(0.10 * len(df11)), replace=False)
    df11 = pd.concat([df11, df11.iloc[dup_idx]], ignore_index=True)
    variants["G11_COMBINED_MUTATION"] = df11

    df12 = base_df.copy()
    for col in df12.select_dtypes(include=[np.number]).columns:
        df12[col] = df12[col] * 1000.0
    variants["G12_SCALE_TRANSFORMED"] = df12

    return variants

variants = make_variants(base_df, GLOBAL_SEED)
for name, df in variants.items():
    print(f"{name:30s} -> {df.shape}")

In [ ]:
# Write all variants to Bronze layer
for name, pdf in variants.items():
    pdf_copy = pdf.copy()
    if "transaction_date" in pdf_copy.columns:
        pdf_copy["transaction_date"] = pdf_copy["transaction_date"].astype(str)

    sdf = session.create_dataframe(pdf_copy)
    table_name = f"DATASET_GENOME.BRONZE.{name.lower()}"
    sdf.write.mode("overwrite").save_as_table(table_name)
    print(f"Written: {table_name}  rows={sdf.count()}")

In [ ]:
# Profiling functions
def get_column_types(sdf):
    numeric_cols, categorical_cols, temporal_cols = [], [], []
    for field in sdf.schema.fields:
        dt = field.datatype
        if isinstance(dt, (IntegerType, LongType, DoubleType, FloatType, DecimalType, ShortType)):
            numeric_cols.append(field.name)
        elif isinstance(dt, (TimestampType, DateType)):
            temporal_cols.append(field.name)
        else:
            categorical_cols.append(field.name)
    return numeric_cols, categorical_cols, temporal_cols


def profile_dataset(sdf, dataset_id):
    n_rows = sdf.count()
    n_cols = len(sdf.columns)
    numeric_cols, categorical_cols, temporal_cols = get_column_types(sdf)

    missing_exprs = [F.sum(F.when(F.col(c).is_null(), 1).otherwise(0)).alias(c) for c in sdf.columns]
    missing_row = sdf.select(missing_exprs).collect()[0].as_dict()
    total_missing = sum(missing_row.values())
    missingness = total_missing / (n_rows * n_cols) if n_rows * n_cols > 0 else 0.0

    distinct_rows = sdf.drop_duplicates().count()
    duplicate_rate = (n_rows - distinct_rows) / n_rows if n_rows > 0 else 0.0

    numeric_stats = {}
    for c in numeric_cols[:20]:
        try:
            stats_row = sdf.select(
                F.mean(c).alias("mean"), F.stddev(c).alias("std"),
                F.min(c).alias("min"), F.max(c).alias("max")
            ).collect()[0]
            numeric_stats[c] = {
                "mean": float(stats_row["MEAN"]) if stats_row["MEAN"] is not None else 0.0,
                "std": float(stats_row["STD"]) if stats_row["STD"] is not None else 1.0,
                "min": float(stats_row["MIN"]) if stats_row["MIN"] is not None else 0.0,
                "max": float(stats_row["MAX"]) if stats_row["MAX"] is not None else 0.0,
            }
        except Exception:
            numeric_stats[c] = {"mean": 0.0, "std": 1.0, "min": 0.0, "max": 0.0}

    cat_stats = {}
    for c in categorical_cols[:10]:
        try:
            vc = sdf.group_by(c).count().sort(F.desc("COUNT")).limit(20).collect()
            total = sum([r["COUNT"] for r in vc]) if vc else 1
            cat_stats[c] = {
                "n_unique": sdf.select(c).distinct().count(),
                "top_values": [(r[c], r["COUNT"] / total) for r in vc],
            }
        except Exception:
            cat_stats[c] = {"n_unique": 1, "top_values": []}

    return {
        "dataset_id": dataset_id, "rows": n_rows, "columns": n_cols,
        "numeric_columns": numeric_cols, "categorical_columns": categorical_cols,
        "temporal_columns": temporal_cols, "missingness": missingness,
        "duplicate_rate": duplicate_rate, "numeric_stats": numeric_stats,
        "cat_stats": cat_stats,
    }

print("Profiling functions defined.")

In [ ]:
# Profile all datasets
profiles = {}
for name in variants.keys():
    sdf = session.table(f"DATASET_GENOME.BRONZE.{name.lower()}")
    prof = profile_dataset(sdf, name)
    profiles[name] = prof
    print(f"Profiled: {name:30s}  rows={prof['rows']}  cols={prof['columns']}  missing={prof['missingness']:.4f}")

profiles_records = [{
    "dataset_id": name, "rows": p["rows"], "columns": p["columns"],
    "n_numeric": len(p["numeric_columns"]),
    "n_categorical": len(p["categorical_columns"]),
    "n_temporal": len(p["temporal_columns"]),
    "missingness": float(p["missingness"]),
    "duplicate_rate": float(p["duplicate_rate"]),
} for name, p in profiles.items()]

profiles_pdf = pd.DataFrame(profiles_records)
session.create_dataframe(profiles_pdf).write.mode("overwrite").save_as_table("DATASET_GENOME.SILVER.DATASET_PROFILES")
print("Silver profiles saved:", profiles_pdf.shape)

In [ ]:
# Dependency analysis
def compute_correlation_matrix(sdf, numeric_cols):
    if len(numeric_cols) < 2:
        return pd.DataFrame()
    pdf = sdf.select(numeric_cols[:15]).sample(0.3).to_pandas()
    pdf = pdf.fillna(pdf.median(numeric_only=True))
    return pdf.corr(method="pearson")


def compute_dependency_graph(corr_matrix, threshold=0.3):
    G = nx.Graph()
    if corr_matrix.empty:
        return G
    cols = corr_matrix.columns.tolist()
    for c in cols:
        G.add_node(c)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            w = abs(corr_matrix.iloc[i, j])
            if w >= threshold:
                G.add_edge(cols[i], cols[j], weight=float(w))
    return G


def graph_features(G):
    n = G.number_of_nodes()
    e = G.number_of_edges()
    density = nx.density(G) if n > 1 else 0.0
    try:
        avg_clust = nx.average_clustering(G, weight="weight") if n > 2 else 0.0
    except Exception:
        avg_clust = 0.0
    n_components = nx.number_connected_components(G) if n > 0 else 0
    return {"n_nodes": n, "n_edges": e, "density": density,
            "avg_clustering": avg_clust, "n_components": n_components}


dependency_data = {}
for name, p in profiles.items():
    sdf = session.table(f"DATASET_GENOME.BRONZE.{name.lower()}")
    corr = compute_correlation_matrix(sdf, p["numeric_columns"])
    G = compute_dependency_graph(corr, threshold=0.3)
    feats = graph_features(G)
    dependency_data[name] = {"corr_matrix": corr, "graph": G, "graph_features": feats}
    print(f"{name:30s} nodes={feats['n_nodes']} edges={feats['n_edges']} density={feats['density']:.3f}")

In [ ]:
# Build genome vectors
def build_schema_genome(p):
    n_cols = p["columns"]
    n_num = len(p["numeric_columns"])
    n_cat = len(p["categorical_columns"])
    n_temp = len(p["temporal_columns"])
    return np.array([n_cols/50.0, n_num/50.0, n_cat/50.0, n_temp/50.0,
                     n_num/max(n_cols,1), n_cat/max(n_cols,1), n_temp/max(n_cols,1)])

def build_statistical_genome(p):
    feats = []
    for c in p["numeric_columns"][:20]:
        s = p["numeric_stats"].get(c, {})
        feats.extend([
            s.get("mean", 0.0) / (abs(s.get("std", 1.0)) + 1e-6),
            s.get("std", 0.0) / (abs(s.get("mean", 1.0)) + 1e-6),
            (s.get("max", 0.0) - s.get("min", 0.0)) / (abs(s.get("mean", 1.0)) + 1e-6),
        ])
    if len(feats) < 60:
        feats.extend([0.0] * (60 - len(feats)))
    return np.array(feats[:60])

def build_categorical_genome(p):
    feats = []
    for c in p["categorical_columns"][:10]:
        cs = p["cat_stats"].get(c, {})
        n_unique = cs.get("n_unique", 1)
        top = cs.get("top_values", [])
        top_p = top[0][1] if top else 0.0
        feats.extend([n_unique / 100.0, top_p, 1.0 - top_p])
    if len(feats) < 30:
        feats.extend([0.0] * (30 - len(feats)))
    return np.array(feats[:30])

def build_dependency_genome(p, dep_data):
    gf = dep_data["graph_features"]
    corr = dep_data["corr_matrix"]
    if corr.empty:
        return np.array([0.0] * 10)
    vals = []
    cols = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            vals.append(abs(corr.iloc[i, j]))
    vals = sorted(vals, reverse=True)[:6]
    while len(vals) < 6:
        vals.append(0.0)
    return np.array([gf["n_nodes"]/50.0, gf["n_edges"]/200.0, gf["density"],
                     gf["avg_clustering"], gf["n_components"]/20.0,
                     vals[0], vals[1], vals[2], vals[3], vals[4]])

def build_quality_genome(p):
    return np.array([p["missingness"], p["duplicate_rate"],
                     1.0 - p["missingness"], 1.0 - min(p["duplicate_rate"], 1.0)])

def build_temporal_genome(p):
    feats = [0.5] * min(len(p["temporal_columns"]), 5)
    while len(feats) < 5:
        feats.append(0.0)
    return np.array(feats)


genome_vectors = {}
genome_components = {}

for name, p in profiles.items():
    gs = build_schema_genome(p)
    gst = build_statistical_genome(p)
    gc = build_categorical_genome(p)
    gd = build_dependency_genome(p, dependency_data[name])
    gq = build_quality_genome(p)
    gt = build_temporal_genome(p)
    genome_components[name] = {"GS": gs, "GST": gst, "GC": gc, "GD": gd, "GQ": gq, "GT": gt}
    genome_vectors[name] = np.concatenate([gs, gst, gc, gd, gq, gt])

GENOME_DIM = len(next(iter(genome_vectors.values())))
print("Genome dimension:", GENOME_DIM)

In [ ]:
# Save genome to Gold
genome_rows = [{"dataset_id": name, **{f"g_{i:03d}": float(v) for i, v in enumerate(vec)}}
               for name, vec in genome_vectors.items()]
genome_pdf = pd.DataFrame(genome_rows)
session.create_dataframe(genome_pdf).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATASET_GENOME")
print("Gold genome saved:", genome_pdf.shape)

In [ ]:
# Similarity matrix
def cosine_sim(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-9 or nb < 1e-9:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

def euclidean_dist(a, b):
    return float(np.linalg.norm(a - b))

names = list(genome_vectors.keys())
n = len(names)
sim_matrix = np.zeros((n, n))
dist_matrix = np.zeros((n, n))
component_sims = {k: np.zeros((n, n)) for k in ["GS", "GST", "GC", "GD", "GQ", "GT"]}

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_sim(genome_vectors[names[i]], genome_vectors[names[j]])
        dist_matrix[i, j] = euclidean_dist(genome_vectors[names[i]], genome_vectors[names[j]])
        for k in component_sims:
            component_sims[k][i, j] = cosine_sim(genome_components[names[i]][k], genome_components[names[j]][k])

sim_df = pd.DataFrame(sim_matrix, index=names, columns=names)
print("Similarity matrix:", sim_df.shape)

In [ ]:
# Save similarity pairs
pairs = []
for i in range(n):
    for j in range(i + 1, n):
        comp_vals = {k: component_sims[k][i, j] for k in component_sims}
        primary = max(comp_vals, key=comp_vals.get)
        pairs.append({
            "dataset_a": names[i], "dataset_b": names[j],
            "genome_similarity": float(sim_matrix[i, j]),
            "distance": float(dist_matrix[i, j]),
            "primary_similarity_component": primary,
            "schema_similarity": float(component_sims["GS"][i, j]),
            "statistical_similarity": float(component_sims["GST"][i, j]),
            "categorical_similarity": float(component_sims["GC"][i, j]),
            "dependency_similarity": float(component_sims["GD"][i, j]),
            "quality_similarity": float(component_sims["GQ"][i, j]),
            "temporal_similarity": float(component_sims["GT"][i, j]),
        })

sim_pairs_df = pd.DataFrame(pairs)
session.create_dataframe(sim_pairs_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_GENOME_SIMILARITY")
print("Gold similarity saved:", sim_pairs_df.shape)

In [ ]:
# Drift detection
baseline = "G01_BASE"
drift_rows = []
for name in names:
    if name == baseline:
        continue
    dist = dist_matrix[names.index(baseline), names.index(name)]
    sim = sim_matrix[names.index(baseline), names.index(name)]
    comp_dists = {k: float(np.linalg.norm(genome_components[baseline][k] - genome_components[name][k]))
                  for k in ["GS", "GST", "GC", "GD", "GQ", "GT"]}
    primary = max(comp_dists, key=comp_dists.get)
    sorted_comps = sorted(comp_dists.items(), key=lambda x: -x[1])
    secondary = sorted_comps[1][0] if len(sorted_comps) > 1 else None
    drift_rows.append({
        "dataset_id": name, "baseline": baseline,
        "genome_distance": dist, "genome_similarity": sim,
        "drift_flag": "DRIFT" if dist > 0.5 else "STABLE",
        "primary_contributor": primary, "secondary_contributor": secondary,
        "GS_dist": comp_dists["GS"], "GST_dist": comp_dists["GST"],
        "GC_dist": comp_dists["GC"], "GD_dist": comp_dists["GD"],
        "GQ_dist": comp_dists["GQ"], "GT_dist": comp_dists["GT"],
    })

drift_df = pd.DataFrame(drift_rows).sort_values("genome_distance", ascending=False)
session.create_dataframe(drift_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_GENOME_DRIFT")
print("Gold drift saved:", drift_df.shape)

In [ ]:
# Quality & Trust Score
WEIGHTS = {
    "missingness": 0.20, "duplicate_rate": 0.10, "outliers": 0.10,
    "invalid_values": 0.15, "redundancy": 0.10, "imbalance": 0.10,
    "dependency_stability": 0.15, "temporal_stability": 0.10,
}

def compute_outlier_rate(sdf, numeric_cols):
    if not numeric_cols:
        return 0.0
    rates = []
    for c in numeric_cols[:10]:
        try:
            q = sdf.approx_quantile(c, [0.25, 0.75], 0.01)
            if len(q) < 2:
                continue
            q1, q3 = q
            iqr = q3 - q1
            if iqr < 1e-9:
                continue
            n_out = sdf.filter((F.col(c) < q1 - 1.5*iqr) | (F.col(c) > q3 + 1.5*iqr)).count()
            total = sdf.count()
            if total > 0:
                rates.append(n_out / total)
        except Exception:
            continue
    return float(np.mean(rates)) if rates else 0.0

quality_rows = []
for name, p in profiles.items():
    sdf = session.table(f"DATASET_GENOME.BRONZE.{name.lower()}")
    outlier_rate = compute_outlier_rate(sdf, p["numeric_columns"])

    imbalance = 0.0
    if "default_flag" in sdf.columns:
        try:
            vc = sdf.group_by("default_flag").count().collect()
            counts = [r["COUNT"] for r in vc]
            if len(counts) == 2 and sum(counts) > 0:
                pmin = min(counts) / sum(counts)
                imbalance = 1.0 - 2 * pmin
        except Exception:
            pass

    redundancy = p["duplicate_rate"]
    dep_dist = np.linalg.norm(genome_components[name]["GD"] - genome_components["G01_BASE"]["GD"])
    dependency_stability = 1.0 / (1.0 + dep_dist)
    temp_dist = np.linalg.norm(genome_components[name]["GT"] - genome_components["G01_BASE"]["GT"])
    temporal_stability = 1.0 / (1.0 + temp_dist)

    W_Q = (WEIGHTS["missingness"] * min(p["missingness"], 1.0) +
           WEIGHTS["duplicate_rate"] * min(redundancy, 1.0) +
           WEIGHTS["outliers"] * min(outlier_rate, 1.0) +
           WEIGHTS["redundancy"] * min(redundancy, 1.0) +
           WEIGHTS["imbalance"] * min(imbalance, 1.0) +
           WEIGHTS["dependency_stability"] * (1.0 - dependency_stability) +
           WEIGHTS["temporal_stability"] * (1.0 - temporal_stability))

    quality_rows.append({
        "dataset_id": name, "missingness": p["missingness"],
        "duplicate_rate": redundancy, "outlier_rate": outlier_rate,
        "invalid_rate": 0.0, "imbalance": imbalance, "redundancy": redundancy,
        "dependency_stability": dependency_stability,
        "temporal_stability": temporal_stability,
        "W_Q": W_Q, "data_trust_score": 100.0 * (1.0 - W_Q),
        "quality_score": 100.0 * (1.0 - (min(p["missingness"],1.0) + min(redundancy,1.0) + min(outlier_rate,1.0) + min(imbalance,1.0)) / 4.0),
    })

quality_df = pd.DataFrame(quality_rows).sort_values("data_trust_score", ascending=False)
session.create_dataframe(quality_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATA_QUALITY")
print("Gold quality saved:", quality_df.shape)

In [ ]:
# Readiness
def readiness_status(dts):
    if dts >= 80: return "READY"
    elif dts >= 60: return "REVIEW"
    else: return "INVESTIGATE"

readiness_rows = []
for _, row in quality_df.iterrows():
    name = row["dataset_id"]
    issue_map = {
        "Missingness": row["missingness"], "Duplicates": row["duplicate_rate"],
        "Outliers": row["outlier_rate"], "Imbalance": row["imbalance"],
        "Redundancy": row["redundancy"],
        "Dependency instability": 1.0 - row["dependency_stability"],
        "Temporal instability": 1.0 - row["temporal_stability"],
    }
    sorted_issues = sorted(issue_map.items(), key=lambda x: -x[1])
    readiness_rows.append({
        "dataset_id": name, "data_trust_score": row["data_trust_score"],
        "schema_stability": float(1.0 / (1.0 + np.linalg.norm(genome_components[name]["GS"] - genome_components["G01_BASE"]["GS"]))),
        "statistical_stability": float(1.0 / (1.0 + np.linalg.norm(genome_components[name]["GST"] - genome_components["G01_BASE"]["GST"]))),
        "dependency_stability": row["dependency_stability"],
        "quality_score": row["quality_score"],
        "temporal_stability": row["temporal_stability"],
        "readiness_status": readiness_status(row["data_trust_score"]),
        "primary_issue": sorted_issues[0][0],
        "secondary_issue": sorted_issues[1][0],
    })

readiness_df = pd.DataFrame(readiness_rows).sort_values("data_trust_score", ascending=False)
session.create_dataframe(readiness_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATASET_READINESS")
print("Gold readiness saved:", readiness_df.shape)

In [ ]:
# Risk profile
risk_rows = []
for _, row in quality_df.iterrows():
    name = row["dataset_id"]
    comp = genome_components[name]
    base = genome_components["G01_BASE"]
    quality_risk = min(1.0, row["W_Q"])
    drift_risk = min(1.0, np.linalg.norm(genome_vectors[name] - genome_vectors["G01_BASE"]) / 5.0)
    dependency_risk = min(1.0, np.linalg.norm(comp["GD"] - base["GD"]) / 2.0)
    redundancy_risk = min(1.0, row["redundancy"] * 2.0)
    schema_risk = min(1.0, np.linalg.norm(comp["GS"] - base["GS"]) / 2.0)
    temporal_risk = min(1.0, np.linalg.norm(comp["GT"] - base["GT"]) / 2.0)
    risk_rows.append({
        "dataset_id": name, "quality_risk": quality_risk, "drift_risk": drift_risk,
        "dependency_risk": dependency_risk, "redundancy_risk": redundancy_risk,
        "schema_risk": schema_risk, "temporal_risk": temporal_risk,
        "overall_risk": (quality_risk + drift_risk + dependency_risk + redundancy_risk + schema_risk + temporal_risk) / 6.0,
    })

risk_df = pd.DataFrame(risk_rows).sort_values("overall_risk", ascending=False)
session.create_dataframe(risk_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATASET_RISK")
print("Gold risk saved:", risk_df.shape)

In [ ]:
# Redundancy
redundancy_rows = []
for _, row in sim_pairs_df.iterrows():
    flag = "POTENTIAL_REDUNDANCY" if (
        row["genome_similarity"] > 0.995 and
        row["schema_similarity"] > 0.99 and
        row["dependency_similarity"] > 0.95 and
        row["categorical_similarity"] > 0.95
    ) else "NONE"
    redundancy_rows.append({
        "dataset_a": row["dataset_a"], "dataset_b": row["dataset_b"],
        "schema_similarity": row["schema_similarity"],
        "statistical_similarity": row["statistical_similarity"],
        "dependency_similarity": row["dependency_similarity"],
        "categorical_similarity": row["categorical_similarity"],
        "overall_similarity": row["genome_similarity"],
        "potential_redundancy_flag": flag,
    })

redundancy_df = pd.DataFrame(redundancy_rows).sort_values("overall_similarity", ascending=False)
session.create_dataframe(redundancy_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATASET_REDUNDANCY")
print("Redundancy candidates:", (redundancy_df["potential_redundancy_flag"] == "POTENTIAL_REDUNDANCY").sum())

In [ ]:
# Portfolio
portfolio_rows = []
for name in names:
    q = quality_df[quality_df["dataset_id"] == name].iloc[0]
    r = risk_df[risk_df["dataset_id"] == name].iloc[0]
    d = drift_df[drift_df["dataset_id"] == name]
    drift_score = float(d["genome_distance"].iloc[0]) if len(d) > 0 else 0.0
    rd = redundancy_df[(redundancy_df["dataset_a"] == name) | (redundancy_df["dataset_b"] == name)]
    redun_score = float(rd["overall_similarity"].max()) if len(rd) > 0 else 0.0
    complexity = float(np.linalg.norm(genome_components[name]["GD"]) + np.linalg.norm(genome_components[name]["GST"]) / 10.0)
    readiness = readiness_df[readiness_df["dataset_id"] == name]["readiness_status"].iloc[0]

    portfolio_rows.append({
        "dataset_id": name, "domain": "Synthetic",
        "trust_score": float(q["data_trust_score"]),
        "drift_score": drift_score, "redundancy_score": redun_score,
        "complexity_score": complexity, "readiness": readiness,
        "overall_risk": float(r["overall_risk"]),
    })

portfolio_df = pd.DataFrame(portfolio_rows)
session.create_dataframe(portfolio_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATASET_PORTFOLIO")
print("Portfolio saved:", portfolio_df.shape)

In [ ]:
# Business Asset Catalog
domains = ["Finance", "Marketing", "Sales", "Operations", "Risk", "Customer Analytics", "Supply Chain"]
report_templates = [
    "Monthly {d} Dashboard", "Quarterly {d} Report", "Daily {d} Monitor",
    "Annual {d} Summary", "Ad-hoc {d} Analysis",
]
owners = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace"]
criticalities = ["High", "Medium", "Low"]

rng = np.random.RandomState(GLOBAL_SEED)
catalog_rows = []
for ds in names:
    n_reports = rng.randint(1, 3)
    for _ in range(n_reports):
        d = rng.choice(domains)
        rname = rng.choice(report_templates).format(d=d)
        catalog_rows.append({
            "dataset_id": ds, "business_domain": d,
            "report_name": rname, "business_owner": rng.choice(owners),
            "criticality": rng.choice(criticalities, p=[0.3, 0.5, 0.2]),
        })

catalog_df = pd.DataFrame(catalog_rows)
session.create_dataframe(catalog_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_BUSINESS_ASSET_CATALOG")
print("Business catalog saved:", catalog_df.shape)

In [ ]:
# Report Impact
impact_rows = []
drift_map = dict(zip(drift_df["dataset_id"], drift_df["genome_distance"]))
risk_map = dict(zip(risk_df["dataset_id"], risk_df["overall_risk"]))

for _, row in catalog_df.iterrows():
    ds = row["dataset_id"]
    dd = drift_map.get(ds, 0.0)
    rk = risk_map.get(ds, 0.0)
    risk_level = "HIGH" if (dd > 1.0 or rk > 0.5) else ("MEDIUM" if (dd > 0.3 or rk > 0.3) else "LOW")
    impact_rows.append({
        "dataset_id": ds, "affected_report": row["report_name"],
        "business_domain": row["business_domain"],
        "criticality": row["criticality"],
        "drift_distance": dd, "risk_level": risk_level,
    })

impact_df = pd.DataFrame(impact_rows).sort_values(["risk_level", "drift_distance"], ascending=[True, False])
session.create_dataframe(impact_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_REPORT_IMPACT")
print("Report impact saved:", impact_df.shape)

In [ ]:
# Change Impact
def change_impact(base_name, other_name):
    base = genome_components[base_name]
    other = genome_components[other_name]
    def level(dist):
        if dist < 0.1: return "Stable"
        elif dist < 0.5: return "Moderate change"
        else: return "Large change"
    return {
        "dataset_a": base_name, "dataset_b": other_name,
        "schema_change": level(np.linalg.norm(base["GS"] - other["GS"])),
        "statistical_change": level(np.linalg.norm(base["GST"] - other["GST"])),
        "categorical_change": level(np.linalg.norm(base["GC"] - other["GC"])),
        "dependency_change": level(np.linalg.norm(base["GD"] - other["GD"])),
        "quality_change": level(np.linalg.norm(base["GQ"] - other["GQ"])),
        "temporal_change": level(np.linalg.norm(base["GT"] - other["GT"])),
    }

change_rows = [change_impact("G01_BASE", n) for n in names if n != "G01_BASE"]
change_df = pd.DataFrame(change_rows)
session.create_dataframe(change_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATASET_CHANGE_IMPACT")
print("Change impact saved:", change_df.shape)

In [ ]:
# Change Impact
def change_impact(base_name, other_name):
    base = genome_components[base_name]
    other = genome_components[other_name]
    def level(dist):
        if dist < 0.1: return "Stable"
        elif dist < 0.5: return "Moderate change"
        else: return "Large change"
    return {
        "dataset_a": base_name, "dataset_b": other_name,
        "schema_change": level(np.linalg.norm(base["GS"] - other["GS"])),
        "statistical_change": level(np.linalg.norm(base["GST"] - other["GST"])),
        "categorical_change": level(np.linalg.norm(base["GC"] - other["GC"])),
        "dependency_change": level(np.linalg.norm(base["GD"] - other["GD"])),
        "quality_change": level(np.linalg.norm(base["GQ"] - other["GQ"])),
        "temporal_change": level(np.linalg.norm(base["GT"] - other["GT"])),
    }

change_rows = [change_impact("G01_BASE", n) for n in names if n != "G01_BASE"]
change_df = pd.DataFrame(change_rows)
session.create_dataframe(change_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATASET_CHANGE_IMPACT")
print("Change impact saved:", change_df.shape)

In [ ]:
# Complexity
complexity_rows = []
for name in names:
    p = profiles[name]
    gf = dependency_data[name]["graph_features"]
    cardinality = sum(cs.get("n_unique", 0) for cs in p["cat_stats"].values())
    complexity = (p["columns"] / 50.0 + cardinality / 500.0 +
                  gf["density"] + gf["n_edges"] / 200.0 +
                  len(p["temporal_columns"]) / 5.0)
    complexity_rows.append({
        "dataset_id": name, "n_columns": p["columns"],
        "total_cardinality": cardinality, "dependency_density": gf["density"],
        "graph_edges": gf["n_edges"], "temporal_columns": len(p["temporal_columns"]),
        "complexity_score": complexity,
    })

complexity_df = pd.DataFrame(complexity_rows).sort_values("complexity_score", ascending=False)
session.create_dataframe(complexity_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATASET_COMPLEXITY")
print("Complexity saved:", complexity_df.shape)

In [ ]:
# Evolution Timeline
evolution_rows = []
rng = np.random.RandomState(GLOBAL_SEED)
base_vec = genome_vectors["G01_BASE"]
for month in range(1, 13):
    noise = rng.normal(0, 0.02 * month / 12.0, len(base_vec))
    vec = base_vec + noise
    dist = float(np.linalg.norm(vec - base_vec))
    evolution_rows.append({
        "dataset_id": "G01_BASE", "period": f"2023-{month:02d}",
        "month_num": month, "genome_distance": dist,
        "baseline_distance": dist,
        "drift_flag": "DRIFT" if dist > 0.1 else "STABLE",
    })

evolution_df = pd.DataFrame(evolution_rows)
session.create_dataframe(evolution_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATASET_EVOLUTION")
print("Evolution saved:", evolution_df.shape)

In [ ]:
# Data Contract
expected_genome = genome_vectors["G01_BASE"]
contract_rows = []
for name in names:
    actual = genome_vectors[name]
    dist = euclidean_dist(expected_genome, actual)
    status = "PASS" if dist < 0.3 else ("REVIEW" if dist < 0.8 else "FAIL")
    contract_rows.append({
        "dataset_id": name, "expected_genome": "G01_BASE_expected",
        "actual_genome": name, "distance": dist, "contract_status": status,
    })

contract_df = pd.DataFrame(contract_rows).sort_values("distance")
session.create_dataframe(contract_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_DATA_CONTRACT")
print("Data contract saved:", contract_df.shape)

In [ ]:
# PCA + Clustering + KPIs
X = np.array([genome_vectors[n] for n in names])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=min(5, X_scaled.shape[0], X_scaled.shape[1]))
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca[:, :2], columns=["PC1", "PC2"])
pca_df["dataset_id"] = names
pca_df["group"] = [n.split("_")[0] for n in names]
print("Explained variance:", pca.explained_variance_ratio_)

km = KMeans(n_clusters=4, random_state=GLOBAL_SEED, n_init=10)
labels = km.fit_predict(X_scaled)
cluster_df = pd.DataFrame({"dataset_id": names, "cluster": labels})
try:
    sil = silhouette_score(X_scaled, labels)
except Exception:
    sil = 0.0
print("Silhouette score:", round(sil, 4))

session.create_dataframe(cluster_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_GENOME_EVALUATION")

kpis = {
    "total_datasets": len(names),
    "total_rows": int(sum(profiles[n]["rows"] for n in names)),
    "total_columns": int(sum(profiles[n]["columns"] for n in names)),
    "avg_quality": float(quality_df["quality_score"].mean()),
    "avg_similarity": float(sim_pairs_df["genome_similarity"].mean()),
    "drift_events": int((drift_df["drift_flag"] == "DRIFT").sum()),
    "potential_redundancy": int((redundancy_df["potential_redundancy_flag"] == "POTENTIAL_REDUNDANCY").sum()),
    "datasets_requiring_review": int((readiness_df["readiness_status"] != "READY").sum()),
}
kpi_df = pd.DataFrame([kpis])
session.create_dataframe(kpi_df).write.mode("overwrite").save_as_table("DATASET_GENOME.GOLD.GOLD_KPI_SUMMARY")
print("KPIs:")
for k, v in kpis.items():
    print(f"  {k}: {v}")

In [ ]:
fig = px.imshow(
    sim_df.values, x=sim_df.columns, y=sim_df.index,
    color_continuous_scale="Viridis",
    title="Chart 1 — Dataset Genome Similarity Heatmap",
    labels=dict(color="Cosine Similarity"),
)
fig.update_layout(height=700, width=900)
fig.show()

In [ ]:
fig = px.bar(
    drift_df.sort_values("genome_distance", ascending=False),
    x="dataset_id", y="genome_distance", color="primary_contributor",
    title="Chart 2 — Genome Drift Distance from G01_BASE",
)
fig.update_layout(height=500, width=1000, xaxis_tickangle=-45)
fig.show()

In [ ]:
merged = quality_df.merge(risk_df, on="dataset_id")
fig = px.scatter(
    merged, x="data_trust_score", y="overall_risk",
    text="dataset_id", size="redundancy",
    title="Chart 3 — Data Trust Score vs Overall Risk",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=600, width=900)
fig.show()

In [ ]:
fig = px.scatter(
    portfolio_df, x="trust_score", y="drift_score",
    text="dataset_id", size="complexity_score", color="readiness",
    title="Chart 4 — Portfolio: Trust vs Drift",
)
fig.update_traces(textposition="top center")
fig.add_hline(y=portfolio_df["drift_score"].median(), line_dash="dash", line_color="gray")
fig.add_vline(x=portfolio_df["trust_score"].median(), line_dash="dash", line_color="gray")
fig.update_layout(height=600, width=900)
fig.show()

In [ ]:
fig = px.scatter(
    pca_df, x="PC1", y="PC2", text="dataset_id", color="group",
    title="Chart 5 — PCA of Dataset Genomes",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=600, width=900)
fig.show()

In [ ]:
fig = px.line(
    evolution_df, x="period", y="genome_distance", markers=True,
    title="Chart 6 — Dataset Evolution Timeline (G01_BASE)",
)
fig.update_layout(height=500, width=900)
fig.show()

In [ ]:
fig = px.histogram(
    drift_df, x="genome_distance", nbins=12,
    title="Chart 7 — Histogram: Genome Distance Distribution",
    color_discrete_sequence=["#636EFA"],
)
fig.add_vline(x=drift_df["genome_distance"].mean(), line_dash="dash", line_color="red")
fig.update_layout(height=500, width=900)
fig.show()

In [ ]:
fig = px.bar(
    quality_df.sort_values("quality_score", ascending=False),
    x="dataset_id", y="quality_score", color="quality_score",
    color_continuous_scale="RdYlGn", text="quality_score",
    title="Chart 8 — Quality Score by Dataset",
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.update_layout(height=550, width=1000, xaxis_tickangle=-45)
fig.show()

In [ ]:
fig = px.bar(
    quality_df.sort_values("data_trust_score", ascending=False),
    x="dataset_id", y="data_trust_score", color="data_trust_score",
    color_continuous_scale="Blues", text="data_trust_score",
    title="Chart 9 — Data Trust Score",
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig.add_hline(y=80, line_dash="dot", line_color="green", annotation_text="READY (80)")
fig.add_hline(y=60, line_dash="dot", line_color="orange", annotation_text="REVIEW (60)")
fig.update_layout(height=550, width=1000, xaxis_tickangle=-45)
fig.show()

In [ ]:
comp_cols = ["GS_dist", "GST_dist", "GC_dist", "GD_dist", "GQ_dist", "GT_dist"]
comp_labels = {"GS_dist": "Schema", "GST_dist": "Statistical", "GC_dist": "Categorical",
               "GD_dist": "Dependency", "GQ_dist": "Quality", "GT_dist": "Temporal"}
melt_df = drift_df.melt(id_vars=["dataset_id"], value_vars=comp_cols,
                        var_name="component", value_name="distance")
melt_df["component"] = melt_df["component"].map(comp_labels)

fig = px.bar(
    melt_df, x="dataset_id", y="distance", color="component",
    barmode="stack", title="Chart 10 — Drift Decomposition",
)
fig.update_layout(height=600, width=1100, xaxis_tickangle=-45)
fig.show()

In [ ]:
quality_dims = ["missingness", "duplicate_rate", "outlier_rate", "imbalance", "redundancy"]
melt_q = quality_df.melt(id_vars=["dataset_id"], value_vars=quality_dims,
                         var_name="dimension", value_name="rate")
fig = px.bar(
    melt_q, x="dataset_id", y="rate", color="dimension", barmode="group",
    title="Chart 11 — Quality Dimensions Across Datasets",
)
fig.update_layout(height=600, width=1200, xaxis_tickangle=-45)
fig.show()

In [ ]:
fig = px.histogram(
    quality_df, x="missingness", nbins=10, text_auto=True,
    color_discrete_sequence=["#EF553B"],
    title="Chart 12 — Missingness Distribution",
)
fig.update_layout(height=500, width=900)
fig.show()

In [ ]:
fig = px.histogram(
    sim_pairs_df, x="genome_similarity", nbins=20,
    color_discrete_sequence=["#00CC96"],
    title="Chart 13 — Pairwise Similarity Distribution",
)
fig.add_vline(x=sim_pairs_df["genome_similarity"].mean(), line_dash="dash", line_color="red")
fig.update_layout(height=500, width=900)
fig.show()

In [ ]:
risk_cols = ["quality_risk", "drift_risk", "dependency_risk", "redundancy_risk", "schema_risk", "temporal_risk"]
melt_r = risk_df.melt(id_vars=["dataset_id"], value_vars=risk_cols,
                      var_name="risk_type", value_name="risk_value")
fig = px.bar(
    melt_r, x="dataset_id", y="risk_value", color="risk_type", barmode="stack",
    title="Chart 14 — Risk Profile Decomposition",
)
fig.update_layout(height=600, width=1100, xaxis_tickangle=-45)
fig.show()

In [ ]:
readiness_counts = readiness_df["readiness_status"].value_counts().reset_index()
readiness_counts.columns = ["status", "count"]
fig = px.bar(
    readiness_counts, x="status", y="count", color="status",
    color_discrete_map={"READY": "green", "REVIEW": "orange", "INVESTIGATE": "red"},
    text="count", title="Chart 15 — Readiness Status Counts",
)
fig.update_traces(textposition="outside")
fig.update_layout(height=500, width=700, showlegend=False)
fig.show()

In [ ]:
merged_cq = complexity_df.merge(quality_df[["dataset_id", "quality_score"]], on="dataset_id")
fig = px.scatter(
    merged_cq, x="complexity_score", y="quality_score", text="dataset_id",
    color="quality_score", color_continuous_scale="RdYlGn",
    title="Chart 16 — Complexity vs Quality",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=600, width=900)
fig.show()

In [ ]:
num_cols = base_df.select_dtypes(include=[np.number]).columns.tolist()
n_cols_grid = 4
n_rows_grid = math.ceil(len(num_cols) / n_cols_grid)

fig = make_subplots(rows=n_rows_grid, cols=n_cols_grid,
                    subplot_titles=num_cols,
                    vertical_spacing=0.06, horizontal_spacing=0.05)
for i, col in enumerate(num_cols):
    r, c = i // n_cols_grid + 1, i % n_cols_grid + 1
    fig.add_trace(go.Histogram(x=base_df[col].dropna(), nbinsx=40,
                                marker_color="#636EFA", name=col), row=r, col=c)
fig.update_layout(height=350 * n_rows_grid, width=1400, showlegend=False,
                  title_text="Chart 17 — Histogram Grid: All Numeric Columns (G01_BASE)")
fig.show()

In [ ]:
fig = make_subplots(rows=n_rows_grid, cols=n_cols_grid,
                    subplot_titles=num_cols,
                    vertical_spacing=0.06, horizontal_spacing=0.05)
for i, col in enumerate(num_cols):
    r, c = i // n_cols_grid + 1, i % n_cols_grid + 1
    fig.add_trace(go.Box(y=base_df[col].dropna(), name=col,
                          marker_color="#EF553B", boxmean="sd"), row=r, col=c)
fig.update_layout(height=350 * n_rows_grid, width=1400, showlegend=False,
                  title_text="Chart 18 — Box Plot Grid: All Numeric Columns (G01_BASE)")
fig.show()

In [ ]:
corr_full = base_df[num_cols].corr(method="pearson")
fig = px.imshow(corr_full, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                text_auto=".2f", aspect="auto",
                title="Chart 19 — Full Correlation Heatmap (G01_BASE)",
                labels=dict(color="Pearson r"))
fig.update_layout(height=900, width=1100)
fig.update_xaxes(tickangle=-45)
fig.show()

In [ ]:
key_cols = ["income", "transaction_amount", "credit_score", "account_balance"]
long_rows = []
for name, pdf in variants.items():
    for col in key_cols:
        if col in pdf.columns:
            vals = pdf[col].dropna().sample(min(1500, len(pdf)), random_state=GLOBAL_SEED)
            for v in vals:
                long_rows.append({"dataset_id": name, "feature": col, "value": float(v)})

long_df = pd.DataFrame(long_rows)
fig = make_subplots(rows=1, cols=len(key_cols), subplot_titles=key_cols)
colors = px.colors.qualitative.Set3
for i, col in enumerate(key_cols):
    sub = long_df[long_df["feature"] == col]
    for j, ds in enumerate(sub["dataset_id"].unique()):
        fig.add_trace(go.Violin(y=sub[sub["dataset_id"] == ds]["value"],
                                 name=ds, box_visible=True, meanline_visible=True,
                                 line_color=colors[j % len(colors)],
                                 showlegend=(i == 0), scalegroup=ds),
                      row=1, col=i + 1)
fig.update_layout(height=600, width=1500, violinmode="group",
                  title_text="Chart 20 — Violin Plots: Key Distributions Across Datasets")
fig.show()

In [ ]:
cat_cols = base_df.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [c for c in cat_cols if c != "customer_id"]
n_cols_grid = 3
n_rows_grid = math.ceil(len(cat_cols) / n_cols_grid)

fig = make_subplots(rows=n_rows_grid, cols=n_cols_grid,
                    subplot_titles=cat_cols,
                    vertical_spacing=0.08, horizontal_spacing=0.06)
palette = px.colors.qualitative.Plotly
for i, col in enumerate(cat_cols):
    r, c = i // n_cols_grid + 1, i % n_cols_grid + 1
    vc = base_df[col].value_counts().reset_index()
    vc.columns = [col, "count"]
    fig.add_trace(go.Bar(x=vc[col].astype(str), y=vc["count"],
                          marker_color=palette[i % len(palette)],
                          text=vc["count"], textposition="outside"), row=r, col=c)
fig.update_layout(height=350 * n_rows_grid, width=1400, showlegend=False,
                  title_text="Chart 21 — Categorical Distributions (G01_BASE)")
fig.update_xaxes(tickangle=-30)
fig.show()

In [ ]:
drift_sorted = drift_df.sort_values("genome_distance", ascending=False).reset_index(drop=True)
fig = px.line(drift_sorted, x="dataset_id", y="genome_distance", markers=True,
              title="Chart 22 — Drift Distance Trend (Ranked)")
fig.update_traces(line_color="#EF553B", marker=dict(size=10))
fig.update_layout(height=500, width=1000, xaxis_tickangle=-45)
fig.show()

In [ ]:
SELECT 
    "dataset_id", 
    "rows", 
    "columns", 
    "n_numeric"     AS numeric_columns,
    "n_categorical" AS categorical_columns, 
    "n_temporal"    AS temporal_columns,
    "missingness", 
    (100 - "missingness" * 100) AS quality_score
FROM DATASET_GENOME.SILVER.DATASET_PROFILES
ORDER BY "dataset_id";

In [ ]:
SELECT 
    "dataset_a", 
    "dataset_b",
    ROUND("genome_similarity", 4) AS genome_similarity,
    ROUND("distance", 4)          AS distance,
    "primary_similarity_component"
FROM DATASET_GENOME.GOLD.GOLD_GENOME_SIMILARITY
ORDER BY "genome_similarity" DESC
LIMIT 10;

In [ ]:
SELECT 
    "dataset_a", 
    "dataset_b",
    ROUND("schema_similarity", 4)       AS schema_similarity,
    ROUND("statistical_similarity", 4)  AS statistical_similarity,
    ROUND("dependency_similarity", 4)   AS dependency_similarity,
    ROUND("overall_similarity", 4)      AS overall_similarity
FROM DATASET_GENOME.GOLD.GOLD_DATASET_REDUNDANCY
WHERE "potential_redundancy_flag" = 'POTENTIAL_REDUNDANCY'
ORDER BY "overall_similarity" DESC;

In [ ]:
SELECT 
    "dataset_id", 
    "missingness", 
    "duplicate_rate", 
    "outlier_rate",
    "imbalance", 
    "redundancy", 
    ROUND("quality_score", 2) AS quality_score
FROM DATASET_GENOME.GOLD.GOLD_DATA_QUALITY
ORDER BY "quality_score" ASC;

In [ ]:
SELECT 
    "dataset_id", 
    ROUND("data_trust_score", 2) AS data_trust_score,
    "readiness_status", 
    "primary_issue", 
    "secondary_issue"
FROM DATASET_GENOME.GOLD.GOLD_DATASET_READINESS
ORDER BY "data_trust_score" DESC;

In [ ]:
SELECT 
    "dataset_id", 
    ROUND("genome_distance", 4) AS genome_distance,
    "drift_flag", 
    "primary_contributor", 
    "secondary_contributor"
FROM DATASET_GENOME.GOLD.GOLD_GENOME_DRIFT
ORDER BY "genome_distance" DESC;

In [ ]:
SELECT 
    "dataset_id", 
    ROUND("quality_risk", 3)    AS quality_risk,
    ROUND("drift_risk", 3)      AS drift_risk, 
    ROUND("dependency_risk", 3) AS dependency_risk,
    ROUND("redundancy_risk", 3) AS redundancy_risk,
    ROUND("overall_risk", 3)    AS overall_risk
FROM DATASET_GENOME.GOLD.GOLD_DATASET_RISK
ORDER BY "overall_risk" DESC;

In [ ]:
SELECT 
    "dataset_id", 
    "affected_report", 
    "business_domain",
    "criticality", 
    ROUND("drift_distance", 4) AS drift_distance, 
    "risk_level"
FROM DATASET_GENOME.GOLD.GOLD_REPORT_IMPACT
ORDER BY "drift_distance" DESC;

In [ ]:
SELECT 
    "dataset_id", 
    "expected_genome", 
    "actual_genome",
    ROUND("distance", 4) AS distance, 
    "contract_status"
FROM DATASET_GENOME.GOLD.GOLD_DATA_CONTRACT
ORDER BY "distance" ASC;

In [ ]:
SELECT 'GOLD_DATASET_GENOME' AS table_name, COUNT(*) AS row_count 
FROM DATASET_GENOME.GOLD.GOLD_DATASET_GENOME
UNION ALL
SELECT 'GOLD_GENOME_SIMILARITY', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_GENOME_SIMILARITY
UNION ALL
SELECT 'GOLD_GENOME_DRIFT', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_GENOME_DRIFT
UNION ALL
SELECT 'GOLD_DATA_QUALITY', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_DATA_QUALITY
UNION ALL
SELECT 'GOLD_DATASET_READINESS', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_DATASET_READINESS
UNION ALL
SELECT 'GOLD_DATASET_RISK', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_DATASET_RISK
UNION ALL
SELECT 'GOLD_DATASET_PORTFOLIO', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_DATASET_PORTFOLIO
UNION ALL
SELECT 'GOLD_DATASET_REDUNDANCY', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_DATASET_REDUNDANCY
UNION ALL
SELECT 'GOLD_DATASET_COMPLEXITY', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_DATASET_COMPLEXITY
UNION ALL
SELECT 'GOLD_DATASET_EVOLUTION', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_DATASET_EVOLUTION
UNION ALL
SELECT 'GOLD_DATA_CONTRACT', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_DATA_CONTRACT
UNION ALL
SELECT 'GOLD_KPI_SUMMARY', COUNT(*) 
FROM DATASET_GENOME.GOLD.GOLD_KPI_SUMMARY
ORDER BY table_name;

In [ ]:
# Cell 60: Final Project Signature & Summary
print("=" * 72)
print(" " * 18 + "🎓 DATASET GENOME PROJECT 🎓")
print("=" * 72)
print()
print("  PROJECT TITLE")
print("  ─────────────────────────────────────────────────────────────────")
print("  Dataset Genome — Explainable Structural Fingerprinting")
print("  & Data Intelligence Platform")
print()
print("  STUDENT")
print("  ─────────────────────────────────────────────────────────────────")
print("  Name          : Sourish Dey")
print("  Roll Number   : 23051223")
print("  Email         : 23051223@kiit.ac.in")
print("  University    : KIIT University")
print("  Program       : B.Tech Computer Science & Engineering")
print()
print("  PROJECT DETAILS")
print("  ─────────────────────────────────────────────────────────────────")
print("  Primary Platform   : Databricks Free Edition")
print("  Secondary Platform : Snowflake")
print("  Global Seed        : 1223")
print("  Data               : 100% Synthetic")
print("  Benchmark Datasets : 12 (G01 – G12)")
print("  Genome Dimension   : 116 features")
print("  Genome Components  : Schema, Statistical, Categorical,")
print("                       Dependency, Quality, Temporal")
print()
print("  DELIVERABLES")
print("  ─────────────────────────────────────────────────────────────────")
print("  • Bronze Layer     : 12 synthetic datasets")
print("  • Silver Layer     : Profiling & statistics")
print("  • Gold Layer       : 12 analytics tables")
print("  • Visualizations   : 22 charts")
print("  • Business Queries : 9 SQL queries")
print()

print()
print("=" * 72)
print(" " * 22 + " Sourish Dey | KIIT University")
print("=" * 72)

In [ ]:
-- Final project stamp visible in Snowflake
SELECT 
    'Dataset Genome'                                          AS project_name,
    'Sourish Dey'                                             AS author,
    '23051223'                                                AS roll_number,
    'KIIT University'                                         AS university,
    '23051223@kiit.ac.in'                                     AS email,
    'Databricks + Snowflake'                                  AS platforms,
    1223                                                      AS global_seed,
    CURRENT_TIMESTAMP()                                       AS completed_at
;